# Search-13a — Traverser des murs : chemins minimaux certifiés

**Navigation** : [Search-12a](Search-12a-Composer-Regards.ipynb) | [Index](README.md) | [Version Python](Search-13a-Traverser-Murs-Certifies.ipynb)

Sur un pavage hexagonal, deux chambres de sol sont séparées par un mur coûteux. Nous distinguons chemin de coût minimal, épaisseur du mur, coupure de sommets et certificat de potentiels. Les six directions du pavage font écho aux six swaps de l'opération 13 du chantier ICT #12204 ; l'analogie n'est pas une preuve d'isomorphisme entre paysages.

**Objectifs** : construire les chambres, calculer et certifier un chemin, distinguer épaisseur et largeur, analyser une modification et son seuil. **Prérequis** : graphes, BFS, Dijkstra et flot. **Durée** : 45 minutes. Algorithmes pédagogiques BCL ; pas de solveur externe.

## 1. Plateau hexagonal et mur

Les coordonnées axiales `(q,r)` satisfont `max(|q|,|r|,|q+r|) ≤ 4`. Le mur est composé de deux colonnes et d'une palissade ; entrer en sol coûte 1, entrer dans un mur 3. Les six directions définissent les arêtes.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Text;
using System.Globalization;
using Microsoft.DotNet.Interactive.Formatting;
CultureInfo.DefaultThreadCurrentCulture=CultureInfo.InvariantCulture;
Console.WriteLine($"Runtime {System.Runtime.InteropServices.RuntimeInformation.FrameworkDescription} ; BCL");

Runtime .NET 10.0.14 ; BCL


Le plateau possède 61 salles. Les fonctions suivantes exposent les deux mesures de mur, le plus court chemin et son audit ; les collections sont construites localement et le mur peut être copié avant une expérience.

In [2]:
#nullable enable
using P = (int q,int r);
record ResultatHex(Dictionary<P,double> Dist,Dictionary<P,P> Parents,int Expansions);
record SvgHex(string Content);
static class Hex {
 public static readonly P S=(-4,0),T=(4,-4);
 public static readonly P[] Directions={(1,0),(1,-1),(0,-1),(-1,0),(-1,1),(0,1)};
 public static readonly List<P> Cases=(from q in Enumerable.Range(-4,9) from r in Enumerable.Range(-4,9) where Math.Max(Math.Max(Math.Abs(q),Math.Abs(r)),Math.Abs(q+r))<=4 select(q,r)).ToList();
 public static readonly HashSet<P> Ensemble=Cases.ToHashSet();
 public static HashSet<P> Mur=Initial();
 static HashSet<P> Initial(){var w=Cases.Where(p=>(p.q==-1&&p.r>=-3)||(p.q==0)||(p.q==2&&p.r>=-4&&p.r<=-3)).ToHashSet();return w;}
 public static IEnumerable<P> Voisins(P u)=>Directions.Select(d=>(u.q+d.q,u.r+d.r)).Where(Ensemble.Contains);
 public static double Cout(P v,double cw,HashSet<P>? mur=null)=>(mur??Mur).Contains(v)?cw:1;
 public static ResultatHex Dijkstra(double cw,bool unitaire=false,bool tout=false,HashSet<P>? mur=null){
  var d=new Dictionary<P,double>{{S,0}};var parent=new Dictionary<P,P>();var q=new PriorityQueue<P,(double,int,int)>();q.Enqueue(S,(0,S.q,S.r));int n=0;
  while(q.TryDequeue(out var u,out var prior)){double du=prior.Item1;if(du>d[u])continue;n++;if(!tout&&u==T)break;foreach(var v in Voisins(u)){double nv=du+(unitaire?1:Cout(v,cw,mur));if(!d.TryGetValue(v,out double old)||nv<old){d[v]=nv;parent[v]=u;q.Enqueue(v,(nv,v.q,v.r));}}}return new(d,parent,n);
 }
 public static List<P> Chemin(Dictionary<P,P> parent){var p=new List<P>{T};while(p[^1]!=S)p.Add(parent[p[^1]]);p.Reverse();return p;}
 public static List<HashSet<P>> Chambres(HashSet<P>? mur=null){mur??=Mur;var vus=new HashSet<P>();var blocs=new List<HashSet<P>>();foreach(var s in Cases){if(mur.Contains(s)||!vus.Add(s))continue;var bloc=new HashSet<P>{s};var q=new Queue<P>();q.Enqueue(s);while(q.Count>0){var u=q.Dequeue();foreach(var v in Voisins(u))if(!mur.Contains(v)&&vus.Add(v)){bloc.Add(v);q.Enqueue(v);}}blocs.Add(bloc);}return blocs;}
 public static int Epaisseur(HashSet<P>? mur=null){mur??=Mur;var d=new Dictionary<P,int>{{S,0}};var q=new LinkedList<P>();q.AddFirst(S);while(q.Count>0){var u=q.First!.Value;q.RemoveFirst();foreach(var v in Voisins(u)){int poids=mur.Contains(v)?1:0,nv=d[u]+poids;if(!d.TryGetValue(v,out int old)||nv<old){d[v]=nv;if(poids==0)q.AddFirst(v);else q.AddLast(v);}}}return d[T];}
 public static bool Separe(IEnumerable<P> coupe){var bloque=coupe.ToHashSet();var vu=new HashSet<P>{S};var q=new Queue<P>();q.Enqueue(S);while(q.Count>0){var u=q.Dequeue();foreach(var v in Voisins(u))if(!bloque.Contains(v)&&vu.Add(v))q.Enqueue(v);}return !vu.Contains(T);}
 sealed class Arc {public int V,Cap,Rev; public Arc(int v,int cap,int rev){V=v;Cap=cap;Rev=rev;}}
 public static int Flot(HashSet<P>? mur=null){
  mur??=Mur;int n=Cases.Count;var idx=Cases.Select((p,i)=>(p,i)).ToDictionary(x=>x.p,x=>x.i);var adj=Enumerable.Range(0,2*n).Select(_=>new List<Arc>()).ToArray();
  void Ajouter(int u,int v,int cap){int iu=adj[u].Count,iv=adj[v].Count;adj[u].Add(new Arc(v,cap,iv));adj[v].Add(new Arc(u,0,iu));}
  foreach(var x in Cases){int i=idx[x];Ajouter(2*i,2*i+1,mur.Contains(x)?1:10000);foreach(var v in Voisins(x))Ajouter(2*i+1,2*idx[v],10000);}
  int source=2*idx[S]+1,cible=2*idx[T],flot=0;
  while(true){var prev=Enumerable.Repeat(-1,2*n).ToArray();var bord=new int[2*n];var q=new Queue<int>();prev[source]=source;q.Enqueue(source);while(q.Count>0&&prev[cible]<0){int u=q.Dequeue();for(int j=0;j<adj[u].Count;j++){var a=adj[u][j];if(a.Cap>0&&prev[a.V]<0){prev[a.V]=u;bord[a.V]=j;q.Enqueue(a.V);if(a.V==cible)break;}}}if(prev[cible]<0)break;
   int delta=int.MaxValue;for(int v=cible;v!=source;v=prev[v])delta=Math.Min(delta,adj[prev[v]][bord[v]].Cap);
   for(int v=cible;v!=source;v=prev[v]){int u=prev[v];var a=adj[u][bord[v]];a.Cap-=delta;adj[v][a.Rev].Cap+=delta;}flot+=delta;
  }return flot;
 }
 public static (int aretes,int violations,int serrees) Audit(Dictionary<P,double> phi,IReadOnlyList<P> chemin,double cw,HashSet<P>? mur=null){int a=0,v=0,t=0;foreach(var u in Cases)foreach(var z in Voisins(u)){a++;if(phi[z]>phi[u]+Cout(z,cw,mur)+1e-9)v++;}foreach(var (u,z) in chemin.Zip(chemin.Skip(1)))if(Math.Abs(phi[z]-phi[u]-Cout(z,cw,mur))<1e-9)t++;return(a,v,t);}
 public static int EpaisseurBellman(HashSet<P>? mur=null){mur??=Mur;var d=Cases.ToDictionary(p=>p,_=>10000);d[S]=0;for(int pass=0;pass<Cases.Count-1;pass++){bool change=false;foreach(var u in Cases)foreach(var v in Voisins(u)){int nv=d[u]+(mur.Contains(v)?1:0);if(nv<d[v]){d[v]=nv;change=true;}}if(!change)break;}return d[T];}
 public static bool CheminValide(IReadOnlyList<P> p)=>p.Count>0&&p[0]==S&&p[^1]==T&&p.All(Ensemble.Contains)&&p.Zip(p.Skip(1),(u,v)=>Voisins(u).Contains(v)).All(x=>x);
 public static double CoutChemin(IReadOnlyList<P> p,double cw,HashSet<P>? mur=null)=>p.Skip(1).Sum(v=>Cout(v,cw,mur));
 public static Dictionary<P,double> Bellman(double cw,HashSet<P>? mur=null){var d=Cases.ToDictionary(p=>p,_=>double.PositiveInfinity);d[S]=0;for(int pass=0;pass<Cases.Count-1;pass++){bool change=false;foreach(var u in Cases)foreach(var v in Voisins(u)){double x=d[u]+Cout(v,cw,mur);if(x<d[v]){d[v]=x;change=true;}}if(!change)break;}return d;}
 public static string Cle(P p)=>$"({p.q},{p.r})";
 public static string Liste(IEnumerable<P> p)=>string.Join(", ",p.Select(Cle));
 public static List<P> Traversee(double cw,HashSet<P>? mur=null){var d=Dijkstra(cw,mur:mur);return Chemin(d.Parents).Where(p=>(mur??Mur).Contains(p)).ToList();}
}
Console.WriteLine("Plateau, BFS, Dijkstra, 0-1 BFS, flot et audit initialises");


Plateau, BFS, Dijkstra, 0-1 BFS, flot et audit initialises


### Carte du plateau

Les coordonnées axiales sont inscrites dans les cases ; vert = sol, beige = mur, S = départ, T = cible. Le fond blanc est inclus dans le SVG.

In [3]:
#nullable enable
Formatter.Register<SvgHex>((v,w)=>w.Write(v.Content),"text/html");
static string Carte(HashSet<(int q,int r)> mur,List<(int q,int r)>? chemin,string titre){
 var sb=new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' width='760' height='700' viewBox='0 0 760 700'><rect width='760' height='700' fill='white'/>");
 sb.Append($"<text x='380' y='28' text-anchor='middle' font-size='19'>{System.Net.WebUtility.HtmlEncode(titre)}</text>");
 (double x,double y) Centre((int q,int r) p)=>(380+32*Math.Sqrt(3)*(p.q+p.r/2.0),310+48*p.r);
 foreach(var p in Hex.Cases){var (x,y)=Centre(p);string points=string.Join(" ",Enumerable.Range(0,6).Select(i=>$"{x+32*Math.Cos(i*Math.PI/3+Math.PI/6):F2},{y+32*Math.Sin(i*Math.PI/3+Math.PI/6):F2}"));string fill=mur.Contains(p)?"#c8b6a0":"#e8f2e8";sb.Append($"<polygon points='{points}' fill='{fill}' stroke='#556' stroke-width='1'/>");}
 if(chemin is not null){string pts=string.Join(" ",chemin.Select(p=>{var (x,y)=Centre(p);return $"{x:F2},{y:F2}";}));sb.Append($"<polyline points='{pts}' fill='none' stroke='#c62828' stroke-width='4'/>");}
 foreach(var p in Hex.Cases){var (x,y)=Centre(p);sb.Append($"<text x='{x:F2}' y='{y+3:F2}' text-anchor='middle' font-size='10' fill='#17212a'>{p.q},{p.r}</text>");}
 foreach(var (p,label,color) in new[]{(Hex.S,"S","#b71c1c"),(Hex.T,"T","#0d47a1")}){var(x,y)=Centre(p);sb.Append($"<circle cx='{x:F2}' cy='{y:F2}' r='10' fill='{color}'/><text x='{x:F2}' y='{y+4:F2}' text-anchor='middle' fill='white' font-size='12'>{label}</text>");}
 sb.Append("<text x='380' y='610' text-anchor='middle' font-size='15'>Coordonnées axiales (q,r) · vert : sol · beige : mur</text><text x='380' y='640' text-anchor='middle' font-size='15'>S : départ · T : cible · ligne rouge : chemin</text></svg>");return sb.ToString();
}
display(new SvgHex(Carte(Hex.Mur,null,"Plateau hexagonal : 61 salles")));
Console.WriteLine($"salles {Hex.Cases.Count}, sol {Hex.Cases.Count-Hex.Mur.Count}, mur {Hex.Mur.Count} ; S={Hex.Cle(Hex.S)}, T={Hex.Cle(Hex.T)} ; couts 1/3");

Plateau hexagonal : 61 salles -4,0 -4,1 -4,2 -4,3 -4,4 -3,-1 -3,0 -3,1 -3,2 -3,3 -3,4 -2,-2 -2,-1 -2,0 -2,1 -2,2 -2,3 -2,4 -1,-3 -1,-2 -1,-1 -1,0 -1,1 -1,2 -1,3 -1,4 0,-4 0,-3 0,-2 0,-1 0,0 0,1 0,2 0,3 0,4 1,-4 1,-3 1,-2 1,-1 1,0 1,1 1,2 1,3 2,-4 2,-3 2,-2 2,-1 2,0 2,1 2,2 3,-4 3,-3 3,-2 3,-1 3,0 3,1 4,-4 4,-3 4,-2 4,-1 4,0 S T Coordonnées axiales (q,r) · vert : sol · beige : mur S : départ · T : cible · ligne rouge : chemin

salles 61, sol 42, mur 19 ; S=(-4,0), T=(4,-4) ; couts 1/3


## 2. Chambres et six directions

Les composantes de sol sont les chambres. La bande de cellules coûteuses les sépare même si un chemin peut traverser le mur en payant son coût d'entrée.

In [4]:
var chambres=Hex.Chambres();if(chambres.Count!=2||chambres.Single(x=>x.Contains(Hex.S)).Contains(Hex.T))throw new Exception("chambres");
Console.WriteLine($"chambres : {chambres.Count} ; S : {chambres.Single(x=>x.Contains(Hex.S)).Count} salles ; T : {chambres.Single(x=>x.Contains(Hex.T)).Count} salles");
int entrees=0,sorties=0;foreach(var d in Hex.Directions){int en=0,so=0;foreach(var u in Hex.Cases){var v=(u.q+d.q,u.r+d.r);if(!Hex.Ensemble.Contains(v))continue;if(!Hex.Mur.Contains(u)&&Hex.Mur.Contains(v))en++;if(Hex.Mur.Contains(u)&&!Hex.Mur.Contains(v))so++;}entrees+=en;sorties+=so;Console.WriteLine($"direction {Hex.Cle(d)} : entrees {en}, sorties {so}");}
if(entrees!=38||sorties!=38)throw new Exception("frontiere");Console.WriteLine($"total : {entrees} entrees et {sorties} sorties");

chambres : 2 ; S : 18 salles ; T : 24 salles


direction (1,0) : entrees 9, sorties 10


direction (1,-1) : entrees 9, sorties 9


direction (0,-1) : entrees 1, sorties 0


direction (-1,0) : entrees 10, sorties 9


direction (-1,1) : entrees 9, sorties 9


direction (0,1) : entrees 0, sorties 1


total : 38 entrees et 38 sorties


On mesure deux chambres de 18 et 24 salles, soit les 42 salles de sol. Les six directions comptent ensemble 38 entrées et 38 sorties de mur ; il s'agit d'arêtes, non de trajets entiers.

## 3. Chemin minimal et comparaison en pas

Dijkstra paie le coût de la cellule dans laquelle il entre. Nous contrôlons le résultat par Bellman–Ford, une vérification indépendante des relaxations, ainsi que chaque arête du chemin.

In [5]:
var d=Hex.Dijkstra(3);var chemin=Hex.Chemin(d.Parents);var mursChemin=chemin.Where(Hex.Mur.Contains).ToList();double L=d.Dist[Hex.T];
var oracle=Hex.Bellman(3);if(!Hex.CheminValide(chemin)||Hex.CoutChemin(chemin,3)!=L||oracle[Hex.T]!=L||L!=12||chemin.Count-1!=8)throw new Exception("chemin ou oracle");
Console.WriteLine($"L={L}, {chemin.Count-1} pas, {mursChemin.Count} murs [{Hex.Liste(mursChemin)}] ; Dijkstra {d.Expansions} expansions ; Bellman-Ford {oracle[Hex.T]}");
display(new SvgHex(Carte(Hex.Mur,chemin,"Chemin minimal S vers T")));

L=12, 8 pas, 2 murs [(-1,-2), (0,-2)] ; Dijkstra 57 expansions ; Bellman-Ford 12


Chemin minimal S vers T -4,0 -4,1 -4,2 -4,3 -4,4 -3,-1 -3,0 -3,1 -3,2 -3,3 -3,4 -2,-2 -2,-1 -2,0 -2,1 -2,2 -2,3 -2,4 -1,-3 -1,-2 -1,-1 -1,0 -1,1 -1,2 -1,3 -1,4 0,-4 0,-3 0,-2 0,-1 0,0 0,1 0,2 0,3 0,4 1,-4 1,-3 1,-2 1,-1 1,0 1,1 1,2 1,3 2,-4 2,-3 2,-2 2,-1 2,0 2,1 2,2 3,-4 3,-3 3,-2 3,-1 3,0 3,1 4,-4 4,-3 4,-2 4,-1 4,0 S T Coordonnées axiales (q,r) · vert : sol · beige : mur S : départ · T : cible · ligne rouge : chemin

Le chemin coûte 12 en huit pas : six entrées de sol et deux de mur. La seconde carte trace la route ; l'égalité avec Bellman–Ford et le contrôle des arêtes ne dépendent pas d'une lecture visuelle.

## 4. Épaisseur et largeur : deux questions

L'épaisseur `m_path` est le minimum de murs traversés par **un** chemin (0–1 BFS). La largeur `m` est la taille minimale d'une coupe de **sommets de mur** : retirer ces sommets et leurs arêtes sépare S de T. Ce retrait n'est pas une transformation du mur en sol. Le flot maximal à nœuds fendus donne une borne inférieure sur la coupe ; une chaîne séparante en donne une borne supérieure.

In [6]:
var chaineA=Hex.Cases.Where(p=>p.q==-1).ToList();var chaineB=Hex.Cases.Where(p=>p.q==0).ToList();int epaisseur=Hex.Epaisseur(),coupe=Hex.Flot();
if(epaisseur!=Hex.EpaisseurBellman()||epaisseur!=2||coupe!=8||!Hex.Separe(chaineA)||!Hex.Separe(chaineB)||chaineA.Count!=8||chaineB.Count!=9||mursChemin.Count<epaisseur)throw new Exception("coupe ou epaisseur");
if(chaineA.Any(cell=>Hex.Separe(chaineA.Where(p=>p!=cell))))throw new Exception("coupe non minimale par inclusion");
Console.WriteLine($"m_path={epaisseur} (0-1 BFS et Bellman-Ford) ; flot/coupe={coupe} ; retrait chaine A ({chaineA.Count}) ou B ({chaineB.Count}) separe S de T ; sept sommets de A ne suffisent pas");

m_path=2 (0-1 BFS et Bellman-Ford) ; flot/coupe=8 ; retrait chaine A (8) ou B (9) separe S de T ; sept sommets de A ne suffisent pas


Ici l'épaisseur vaut 2, alors que la largeur vaut 8 : une route franchit deux couches, mais huit sommets de mur sont nécessaires pour séparer les extrémités. Le flot fournit huit unités disjointes en sommets de mur et la chaîne A fournit une coupe de huit.

## 5. Certificat de potentiels

Un potentiel `φ` avec `φ(S)=0`, `φ(v) ≤ φ(u)+coût(v)` pour chaque arête et égalité sur le chemin fournit une borne inférieure atteinte. Nous auditons toutes les arêtes et vérifions qu'un potentiel corrompu est rejeté.

In [7]:
var complet=Hex.Dijkstra(3,tout:true);var phi=complet.Dist;var audit=Hex.Audit(phi,chemin,3);var mauvais=phi.ToDictionary(x=>x.Key,x=>x.Value);mauvais[Hex.T]+=100;var auditFaux=Hex.Audit(mauvais,chemin,3);
if(phi.Count!=61||audit.aretes!=312||audit.violations!=0||audit.serrees!=8||phi[Hex.S]!=0||phi[Hex.T]!=L||auditFaux.violations==0)throw new Exception("audit");
var pasSeuls=Hex.Dijkstra(3,unitaire:true);var cheminPas=Hex.Chemin(pasSeuls.Parents);double coutPas=Hex.CoutChemin(cheminPas,3);
if(!Hex.CheminValide(cheminPas)||cheminPas.Count-1!=8||coutPas!=14)throw new Exception("contre-exemple pas");
Console.WriteLine($"audit : {audit.aretes} aretes, {audit.violations} violation, {audit.serrees}/{chemin.Count-1} egalites ; potentiel corrompu : {auditFaux.violations} violations");
Console.WriteLine($"route minimale en pas : {cheminPas.Count-1} pas mais cout {coutPas} ; optimum cout {L}");

audit : 312 aretes, 0 violation, 8/8 egalites ; potentiel corrompu : 3 violations


route minimale en pas : 8 pas mais cout 14 ; optimum cout 12


Les 312 inégalités sont satisfaites et les huit arêtes du chemin sont serrées : le coût 12 est certifié. Une route de huit pas peut coûter 14 ; minimiser les pas et minimiser le coût sont deux tâches différentes.

## 6. Modifications et balayage du prix

Percer une case signifie **la transformer en sol** ; c'est une opération différente du retrait de sommet utilisé par la coupe. Une modification qui conserve `L(S,T)` et l'ancien certificat ne prouve pas que toutes les préférences ou le paysage entier restent inchangés.

In [8]:
var perce=Hex.Mur.ToHashSet();var point=mursChemin[0];perce.Remove(point);var dPerce=Hex.Dijkstra(3,mur:perce);var violPerce=Hex.Audit(phi,chemin,3,perce).violations;bool fusion=Hex.Chambres(perce).Any(b=>b.Contains(Hex.S)&&b.Contains(Hex.T));
var feuille=Hex.Mur.ToHashSet();feuille.Add((-2,4));var dFeuille=Hex.Dijkstra(3,mur:feuille);var violFeuille=Hex.Audit(phi,chemin,3,feuille).violations;
if(dPerce.Dist[Hex.T]!=10||violPerce!=2||fusion||dFeuille.Dist[Hex.T]!=12||violFeuille!=0)throw new Exception("modifications");
Console.WriteLine($"percement {Hex.Cle(point)} : L {L}->{dPerce.Dist[Hex.T]}, ancien certificat {violPerce} violations, chambres fusionnees={fusion}");
Console.WriteLine($"case (-2,4) en mur : L={dFeuille.Dist[Hex.T]}, ancien certificat {violFeuille} violation ; isomorphisme global non etabli");
var nombresMurs=new HashSet<int>();for(int cw=1;cw<=9;cw++){var dd=Hex.Dijkstra(cw);var pp=Hex.Chemin(dd.Parents);var traverse=pp.Where(Hex.Mur.Contains).ToList();nombresMurs.Add(traverse.Count);Console.WriteLine($"c_w={cw} : L={dd.Dist[Hex.T]}, {traverse.Count} murs [{Hex.Liste(traverse)}]");}if(nombresMurs.Count<2)throw new Exception("bascule absente");

percement (-1,-2) : L 12->10, ancien certificat 2 violations, chambres fusionnees=False


case (-2,4) en mur : L=12, ancien certificat 0 violation ; isomorphisme global non etabli


c_w=1 : L=8, 3 murs [(-1,-3), (0,-4), (2,-4)]


c_w=2 : L=10, 2 murs [(-1,-2), (0,-2)]


c_w=3 : L=12, 2 murs [(-1,-2), (0,-2)]


c_w=4 : L=14, 2 murs [(-1,-2), (0,-2)]


c_w=5 : L=16, 2 murs [(-1,-2), (0,-2)]


c_w=6 : L=18, 2 murs [(-1,-2), (0,-2)]


c_w=7 : L=20, 2 murs [(-1,-2), (0,-2)]


c_w=8 : L=22, 2 murs [(-1,-2), (0,-2)]


c_w=9 : L=24, 2 murs [(-1,-2), (0,-2)]


Le percement fait passer L de 12 à 10 et invalide l'ancien potentiel par deux violations. La modification de la feuille conserve L et ce certificat S→T ; elle ne démontre pas un isomorphisme global. Au prix entier 2, le passage coûte 10 ; juste au-dessus de 1 son coût est `6 + 2c_w`, proche de 8.

## 7. Exemples guidés et exercices

Les trois exemples résolus ci-dessous reprennent les contributions de **Tristan Dhersin**, [PR #25535](https://github.com/jsboige/CoursIA/pull/25535). Les exercices 1b, 2b et 3b sont distincts et restent à compléter.

### Exemple guidé 1 — une troisième colonne (Tristan Dhersin, #25535)

Ajouter la colonne `q=1`, mesurer l'épaisseur, la coupe et le chemin ; ne pas modifier le plateau de référence.

In [9]:
var trois=Hex.Mur.Union(Hex.Cases.Where(p=>p.q==1)).ToHashSet();int ep3=Hex.Epaisseur(trois),coupe3=Hex.Flot(trois);var d3=Hex.Dijkstra(3,mur:trois);var p3=Hex.Chemin(d3.Parents);var m3=p3.Where(trois.Contains).ToList();
if(ep3!=Hex.EpaisseurBellman(trois)||ep3!=3||coupe3!=8||d3.Dist[Hex.T]!=14||p3.Count-1!=8||m3.Count!=3||Hex.Mur.Count!=19)throw new Exception("exemple mur epais");
Console.WriteLine($"colonne q=1 : {Hex.Cases.Count(p=>p.q==1)} cellules ; epaisseur {ep3}, coupe {coupe3} ; L={d3.Dist[Hex.T]}, {p3.Count-1} pas, murs [{Hex.Liste(m3)}]");

colonne q=1 : 8 cellules ; epaisseur 3, coupe 8 ; L=14, 8 pas, murs [(-1,-2), (0,-2), (1,-2)]


L'épaisseur passe de 2 à 3 ; la coupe reste 8. Le chemin vaut 14 en huit pas et traverse une cellule de chacune des trois colonnes.

### Exercice 1b — ouvrir deux portes

Copier le plateau, transformer `(-1,0)` et `(-1,1)` en sol, puis mesurer épaisseur, coupe et Dijkstra. Retourner le plateau modifié, les mesures et un chemin ; le harnais contrôle aussi une autre ouverture, l'entrée intacte et le coût recalculé.

In [10]:
#nullable enable
record PorteResultat(HashSet<(int q,int r)> Mur,int Epaisseur,int Coupe,double Cout,List<(int q,int r)> Chemin);
static PorteResultat? PorteExercice(HashSet<(int q,int r)> mur,(int q,int r)[] portes){
 // TODO : copier le mur, convertir les portes en sol, mesurer les deux largeurs
 // et retourner aussi un chemin optimal. Ne pas modifier l'entree.
 return null;
}
static PorteResultat? VerifierPorte((int q,int r)[] portes){
 var initial=Hex.Mur.ToHashSet();var copie=initial.ToHashSet();var r=PorteExercice(copie,portes);if(r is null)return null;
 var attendu=initial.Except(portes).ToHashSet();if(!copie.SetEquals(initial)||!r.Mur.SetEquals(attendu)||!Hex.CheminValide(r.Chemin)||
  r.Epaisseur!=Hex.Epaisseur(attendu)||r.Coupe!=Hex.Flot(attendu)||r.Cout!=Hex.Dijkstra(3,mur:attendu).Dist[Hex.T]||
  Hex.CoutChemin(r.Chemin,3,attendu)!=r.Cout)throw new Exception("porte, chemin ou mesures incorrects");return r;
}
var porte=VerifierPorte(new[]{(-1,0),(-1,1)});var porteBis=VerifierPorte(new[]{(-1,-2)});
if(porte is null||porteBis is null)Console.WriteLine("Exercice 1b a completer");
else {if(porte.Epaisseur!=1||porte.Coupe!=9||porte.Cout!=10||!porte.Chemin.Any(p=>p==(-1,0)||p==(-1,1)))throw new Exception("fixture porte");Console.WriteLine($"Exercice 1b : epaisseur {porte.Epaisseur}, coupe {porte.Coupe}, cout {porte.Cout} ; deux modifications controlees");}


Exercice 1b a completer


### Exemple guidé 2 — réauditer à `c_w=5` (Tristan Dhersin, #25535)

Recalculer les potentiels sur les 61 salles et contrôler toutes les arêtes et toutes les égalités du chemin.

In [11]:
var d5=Hex.Dijkstra(5,tout:true);var p5=Hex.Chemin(d5.Parents);var a5=Hex.Audit(d5.Dist,p5,5);var m5=p5.Where(Hex.Mur.Contains).ToList();
if(d5.Dist.Count!=61||d5.Dist[Hex.T]!=16||a5.aretes!=312||a5.violations!=0||a5.serrees!=8||!m5.SequenceEqual(mursChemin))throw new Exception("exemple audit cout 5");
Console.WriteLine($"c_w=5 : L={d5.Dist[Hex.T]}, {p5.Count-1} pas ; audit {a5.aretes} aretes, {a5.violations} violation, {a5.serrees}/{p5.Count-1} egalites ; passage [{Hex.Liste(m5)}]");

c_w=5 : L=16, 8 pas ; audit 312 aretes, 0 violation, 8/8 egalites ; passage [(-1,-2), (0,-2)]


L vaut 16 au lieu de 12 : deux murs coûtent chacun deux unités de plus. Le passage reste identique et les 312 inégalités sont réauditées.

### Exercice 2b — deux routes à égalité

Au prix 1, retourner les potentiels sur toutes les salles et deux chemins optimaux distincts, dont un passe par `(-1,-2)` et `(0,-2)`. Le harnais vérifie 312 arêtes, les égalités de chaque route et leur coût recalculé ; l'optimalité ne prouve pas l'unicité.

In [12]:
#nullable enable
record CertificatEgalite(Dictionary<(int q,int r),double> Phi,List<(int q,int r)> Route1,List<(int q,int r)> Route2);
static CertificatEgalite? EgaliteExercice(){
 // TODO : au prix 1, calculer les potentiels sur toutes les salles et deux
 // chemins optimaux distincts, dont un franchissant (-1,-2),(0,-2).
 return null;
}
var egalite=EgaliteExercice();
if(egalite is null)Console.WriteLine("Exercice 2b a completer");
else {
 if(egalite.Phi.Count!=Hex.Cases.Count||Hex.Cases.Any(p=>!egalite.Phi.ContainsKey(p)||!double.IsFinite(egalite.Phi[p]))||egalite.Phi[Hex.S]!=0||
  !Hex.CheminValide(egalite.Route1)||!Hex.CheminValide(egalite.Route2)||egalite.Route1.SequenceEqual(egalite.Route2)||
  !new[]{egalite.Route1,egalite.Route2}.Any(p=>p.Contains((-1,-2))&&p.Contains((0,-2))))throw new Exception("certificat ou routes");
 foreach(var p in new[]{egalite.Route1,egalite.Route2}){var audit=Hex.Audit(egalite.Phi,p,1);if(audit.aretes!=312||audit.violations!=0||audit.serrees!=p.Count-1||Hex.CoutChemin(p,1)!=egalite.Phi[Hex.T]||egalite.Phi[Hex.T]!=8)throw new Exception("audit a egalite");}
 Console.WriteLine("Exercice 2b : deux routes distinctes certifiees, 312 aretes et egalites verifiees");
}


Exercice 2b a completer


### Exemple guidé 3 — seuil de bascule (Tristan Dhersin, #25535)

Partir de la traversée au coût 1, puis chercher la première valeur entière différente et encadrer le seuil réel par 40 itérations. Comparer aux modèles `5+3c` et `6+2c`.

In [13]:
var reference=Hex.Traversee(1);int lo=1,hi=9;if(Hex.Traversee(hi).SequenceEqual(reference))throw new Exception("bascule");
while(hi-lo>1){int mid=(lo+hi)/2;if(Hex.Traversee(mid).SequenceEqual(reference))lo=mid;else hi=mid;}
double bas=.5,haut=2;for(int i=0;i<40;i++){double mid=(bas+haut)/2;if(Hex.Traversee(mid).SequenceEqual(reference))bas=mid;else haut=mid;}
if(hi!=2||Math.Abs(bas-1)>1e-9||Math.Abs(haut-1)>1e-9)throw new Exception("seuil");
Console.WriteLine($"reference c=1 [{Hex.Liste(reference)}] ; seuil entier {hi}, reel [{bas:F10}, {haut:F10}] en 40 iterations");
foreach(double c in new[]{.5,1,2})Console.WriteLine($"c={c}: palissade={5+3*c}, passage={6+2*c}");

reference c=1 [(-1,-3), (0,-4), (2,-4)] ; seuil entier 2, reel [1.0000000000, 1.0000000000] en 40 iterations


c=0.5: palissade=6.5, passage=7


c=1: palissade=8, passage=8


c=2: palissade=11, passage=10


À 1, les deux routes coûtent 8 ; au-delà, `6+2c` devient moins cher que `5+3c`. Le premier prix entier au-delà du seuil est 2 ; juste au-dessus de 1, le coût n'est pas encore 10.

### Exercice 3b — seuil avec trois colonnes

Écrire une dichotomie utilisant uniquement le callback de traversée : référence à 1, vérification à 9, recherche entière, puis 40 itérations sur `[0,5 ; 2]`. Le harnais rejoue les requêtes réellement observées et teste également un seuil synthétique différent, pour exclure une réponse constante. Les prix interrogés et les bornes retournées doivent être finis ; un oracle entier indépendant vérifie le premier changement.

In [14]:
#nullable enable
record SeuilResultat(int Entier,double Bas,double Haut);
static SeuilResultat? SeuilExercice(Func<double,List<(int q,int r)>> traversee){
 // TODO : reference a 1, verification a 9, dichotomie entiere, puis
 // 40 iterations reelles sur [0.5,2]. Renvoyer entier et encadrement reel.
 return null;
}
static SeuilResultat? VerifierSeuil(Func<double,List<(int q,int r)>> verite,double seuilAttendu){
 var observations=new List<(double c,List<(int q,int r)> route)>();
 List<(int q,int r)> Mesurer(double c){if(!double.IsFinite(c))throw new Exception("prix non fini");var route=verite(c).ToList();observations.Add((c,route));return route.ToList();}
 var resultat=SeuilExercice(Mesurer);if(resultat is null)return null;
 int i=0;List<(int q,int r)> Lecture(double attendu){if(i>=observations.Count||!double.IsFinite(attendu)||!double.IsFinite(observations[i].c)||Math.Abs(observations[i].c-attendu)>1e-12)throw new Exception("mesure manquante ou hors dichotomie");return observations[i++].route;}
 var reference=Lecture(1);if(Lecture(9).SequenceEqual(reference))throw new Exception("pas de changement");int lo=1,hi=9;
 while(hi-lo>1){int mid=(lo+hi)/2;if(Lecture(mid).SequenceEqual(reference))lo=mid;else hi=mid;}
 double bas=.5,haut=2;for(int j=0;j<40;j++){double mid=(bas+haut)/2;if(Lecture(mid).SequenceEqual(reference))bas=mid;else haut=mid;}
 if(!double.IsFinite(resultat.Bas)||!double.IsFinite(resultat.Haut)||!double.IsFinite(seuilAttendu)||resultat.Entier<2||resultat.Entier>9)throw new Exception("seuil non fini ou hors intervalle");
 // Oracle entier separe de la relecture de trace : premier changement sur 1..9.
 int entierOracle=Enumerable.Range(2,8).FirstOrDefault(c=>!verite(c).SequenceEqual(reference));
 if(entierOracle==0 || resultat.Entier!=entierOracle || !verite(resultat.Entier-1).SequenceEqual(reference) || verite(resultat.Entier).SequenceEqual(reference))throw new Exception("seuil entier invalide");
 if(i!=observations.Count||resultat.Entier!=hi||Math.Abs(resultat.Bas-bas)>1e-12||Math.Abs(resultat.Haut-haut)>1e-12||Math.Abs((bas+haut)/2-seuilAttendu)>1e-9)throw new Exception("seuil non mesure");
 return resultat;
}
var troisColonnes=Hex.Mur.Union(Hex.Cases.Where(p=>p.q==1)).ToHashSet();
var seuil3=VerifierSeuil(c=>Hex.Traversee(c,troisColonnes),1);
var seuilTest=VerifierSeuil(c=>c<=1.5?new List<(int,int)>{(0,0)}:new List<(int,int)>{(1,0)},1.5);
if(seuil3 is null||seuilTest is null)Console.WriteLine("Exercice 3b a completer");
else Console.WriteLine($"Exercice 3b : seuil entier {seuil3.Entier}, reel [{seuil3.Bas:F10}, {seuil3.Haut:F10}] ; 40 mesures et fixture independante");


Exercice 3b a completer


## À retenir

La coupe retire des sommets ; le percement les transforme en sol. L'épaisseur minimale vaut 2, la coupe 8 et le chemin de coût 12 est certifié par 312 inégalités. Un ancien certificat valide après modification établit le résultat S→T contrôlé, pas toutes les préférences du paysage.

## Vérifiez votre compréhension

<details><summary>Pourquoi les valeurs 2 et 8 ne se contredisent-elles pas ?</summary>La première compte les murs d'un chemin ; la seconde est la taille d'une coupe de sommets.</details>

<details><summary>Que prouve un potentiel admissible serré sur un chemin ?</summary>Son coût atteint une borne inférieure valable pour tout chemin S→T.</details>

<details><summary>Quel est le coût juste au-dessus de c=1 ?</summary>Pour le passage à deux murs, `6+2c`, proche de 8 ; il vaut 10 à c=2.</details>

## Pour aller plus loin

Lire [Search-12a — Composer des regards](Search-12a-Composer-Regards.ipynb) et les sources de la contribution étudiante [#25535](https://github.com/jsboige/CoursIA/pull/25535). Les exercices 1b, 2b et 3b prolongent les exemples guidés.